# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Low Zhi Yik



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly



# 1. Business Understanding
Goal:  predict resale price of hdb flats in the selected 3 towns


# 2. Data Understanding

## 2.1 Load dataset

Chose 3 towns out from the dataset so that it is faster to train. full dataset has 200000+ rows, too much for the model to train. (Allowed by lecturer)

In [2]:
import pandas as pd

# Load the dataset
FILE_PATH = r"hdb.csv"
df = pd.read_csv(FILE_PATH)

# Extract rows where the town is Sengkang, Pasir Ris and Serangoon
selected_towns = ['SENGKANG', 'PASIR RIS', 'SERANGOON']
df = df[df['town'].str.upper().isin(selected_towns)].copy()
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
670,2017-01,PASIR RIS,3 ROOM,5,CHANGI VILLAGE RD,04 TO 06,66.0,Improved,1981,63 years 06 months,286000.0
671,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0
672,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0
673,2017-01,PASIR RIS,4 ROOM,743,PASIR RIS ST 71,01 TO 03,104.0,Model A,1996,78 years 08 months,377000.0
674,2017-01,PASIR RIS,4 ROOM,109,PASIR RIS ST 11,04 TO 06,103.0,Model A,1990,72 years 08 months,380000.0
...,...,...,...,...,...,...,...,...,...,...,...
205034,2025-04,SERANGOON,EXECUTIVE,537,SERANGOON NTH AVE 4,04 TO 06,147.0,Maisonette,1992,66 years 07 months,1053000.0
205035,2025-04,SERANGOON,EXECUTIVE,525,SERANGOON NTH AVE 4,01 TO 03,150.0,Maisonette,1992,66 years 07 months,1050000.0
205036,2025-05,SERANGOON,EXECUTIVE,506,SERANGOON NTH AVE 4,07 TO 09,146.0,Apartment,1992,66 years 07 months,1000888.0
205037,2025-05,SERANGOON,EXECUTIVE,520,SERANGOON NTH AVE 4,04 TO 06,147.0,Maisonette,1992,66 years 07 months,1058000.0


## 2.2 Summary Statistics

Understand the type of variable for each column

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 26819 entries, 670 to 205038
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                26819 non-null  str    
 1   town                 26819 non-null  str    
 2   flat_type            26819 non-null  str    
 3   block                26819 non-null  str    
 4   street_name          26819 non-null  str    
 5   storey_range         26819 non-null  str    
 6   floor_area_sqm       26819 non-null  float64
 7   flat_model           26819 non-null  str    
 8   lease_commence_date  26819 non-null  int64  
 9   remaining_lease      26819 non-null  str    
 10  resale_price         26819 non-null  float64
dtypes: float64(2), int64(1), str(8)
memory usage: 4.4 MB


Check for number of missing data in selected towns from dataset

In [4]:
df.isna().sum()


month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

General view of the statistics of numerical columns

In [5]:
df.describe()


,floor_area_sqm,lease_commence_date,resale_price
count,26819.000000,26819.000000,2.681900e+04
mean,104.163093,2002.657370,5.344884e+05
std,21.696359,10.441836,1.393362e+05
min,38.000000,1978.000000,1.700000e+05
25%,92.000000,1995.000000,4.300000e+05
50%,104.000000,2001.000000,5.188880e+05
75%,115.000000,2014.000000,6.180000e+05
max,190.000000,2021.000000,1.268000e+06


Check for missing values in the rows of the 3 towns

In [6]:
df.isna().sum()

month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

Find out the number of data is sufficient in each town for effective model results

In [7]:
df['town'].value_counts()


town
SENGKANG     17097
PASIR RIS     6005
SERANGOON     3717
Name: count, dtype: int64

Cleaning data before eda to format some of the categorical data,which that can be converted to numerical data,  into integers or floats.

In [8]:
# Convert storey_range to median storey: 04 TO 06 → 5.0
mid_stories = []
for item in df['storey_range'].str.split(' TO '):
    midpoint = (float(item[0]) + float(item[1])) / 2
    mid_stories.append(midpoint)
df['median_storey'] = mid_stories


# Clean remaining_lease by extracting years and months by splitting the string text

# Loop through every list to calculate the fractional years
clean_lease_years = []

# Split the text by spaces and extracting all data under remaining lease in our dataset
lease_lists = df['remaining_lease'].str.split(' ')

for item in lease_lists:
    # item looks like: ['93', 'years', '04', 'months'] or ['65', 'years']
    years = float(item[0])
    
    # Check if if the data has year and months
    if len(item) > 2:
        months = float(item[2]) / 12.0
    else:
        months = 0.0 #if exact years, months default to 0 as a float
        
    clean_lease_years.append(years + months)

# Assign the final list back to your DataFrame as a new column
df['remaining_lease_years'] = clean_lease_years



# Clean month. Extract just the numeric year ("2017-01" to 2017)
# Grabs the first 4 characters of the text string and changes them to numbers
df['year'] = df['month'].str[:4].astype(int)
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,median_storey,remaining_lease_years,year
670,2017-01,PASIR RIS,3 ROOM,5,CHANGI VILLAGE RD,04 TO 06,66.0,Improved,1981,63 years 06 months,286000.0,5.0,63.500000,2017
671,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0,2.0,78.833333,2017
672,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0,2.0,78.833333,2017
673,2017-01,PASIR RIS,4 ROOM,743,PASIR RIS ST 71,01 TO 03,104.0,Model A,1996,78 years 08 months,377000.0,2.0,78.666667,2017
674,2017-01,PASIR RIS,4 ROOM,109,PASIR RIS ST 11,04 TO 06,103.0,Model A,1990,72 years 08 months,380000.0,5.0,72.666667,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205034,2025-04,SERANGOON,EXECUTIVE,537,SERANGOON NTH AVE 4,04 TO 06,147.0,Maisonette,1992,66 years 07 months,1053000.0,5.0,66.583333,2025
205035,2025-04,SERANGOON,EXECUTIVE,525,SERANGOON NTH AVE 4,01 TO 03,150.0,Maisonette,1992,66 years 07 months,1050000.0,2.0,66.583333,2025
205036,2025-05,SERANGOON,EXECUTIVE,506,SERANGOON NTH AVE 4,07 TO 09,146.0,Apartment,1992,66 years 07 months,1000888.0,8.0,66.583333,2025
205037,2025-05,SERANGOON,EXECUTIVE,520,SERANGOON NTH AVE 4,04 TO 06,147.0,Maisonette,1992,66 years 07 months,1058000.0,5.0,66.583333,2025


## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

### 2.3.1.2 Understanding distribution of features

### 2.3.2 Understanding relationship between variables

# 3. Data Preparation

## 3.1 Data Cleaning

## 3.2 Train-Test Split

# 4. Modelling

### 4.2 Train Model

# 5. Model Evaluation

## Iterative model development
